# LocalLLM — LoRA fine-tuning from the auto-improvement dataset

1-click LoRA training on a **free Colab GPU (T4)** using the reflexion dataset that
`automatic_tester` produces (`datasets/lora_profiles/auto_*.jsonl`, format
`{instruction, input, output}`).

**Runtime → Change runtime type → T4 GPU** before running. Then Runtime → Run all.

In [ ]:
# 1) Install Unsloth (fast LoRA on the free T4)
%pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
%pip install -q --no-deps trl peft accelerate bitsandbytes datasets

In [ ]:
# 2) Provide the dataset. Either upload the auto_*.jsonl file the pipeline made,
#    or set DATASET_URL to a raw link. Uploading is simplest:
from google.colab import files
import glob, os
print('Upload your auto_*.jsonl (from datasets/lora_profiles/) ...')
up = files.upload()
DATASET_PATH = next(iter(up)) if up else sorted(glob.glob('*.jsonl'))[0]
print('Using dataset:', DATASET_PATH)

In [ ]:
# 3) Config — pick the base model you want to improve (must be an Unsloth 4-bit model)
BASE_MODEL   = 'unsloth/Qwen2.5-Coder-7B-Instruct-bnb-4bit'  # or any unsloth/*-bnb-4bit
MAX_SEQ_LEN  = 2048
OUTPUT_NAME  = 'localllm-reflexion'  # name for the resulting adapter / GGUF
MAX_STEPS    = 60   # small dataset -> few steps; raise for larger datasets
LEARNING_RATE = 2e-4

In [ ]:
# 4) Load model + attach LoRA adapters
from unsloth import FastLanguageModel
import torch
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = BASE_MODEL, max_seq_length = MAX_SEQ_LEN,
    dtype = None, load_in_4bit = True)
model = FastLanguageModel.get_peft_model(
    model, r = 16, lora_alpha = 16, lora_dropout = 0,
    target_modules = ['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
    bias = 'none', use_gradient_checkpointing = 'unsloth', random_state = 3407)

In [ ]:
# 5) Load + format the dataset. Tolerates {instruction,input,output} and {prompt,response}.
from datasets import load_dataset
EOS = tokenizer.eos_token
def to_text(ex):
    instr = ex.get('instruction') or ex.get('prompt') or ''
    inp   = ex.get('input') or ''
    out   = ex.get('output') or ex.get('response') or ''
    user  = instr + (('\n\n' + inp) if inp else '')
    msgs  = [{'role':'user','content':user}, {'role':'assistant','content':out}]
    return {'text': tokenizer.apply_chat_template(msgs, tokenize=False) + EOS}
ds = load_dataset('json', data_files=DATASET_PATH, split='train')
ds = ds.map(to_text, remove_columns=ds.column_names)
print('examples:', len(ds)); print(ds[0]['text'][:400])

In [ ]:
# 6) Train
from trl import SFTTrainer
from transformers import TrainingArguments
trainer = SFTTrainer(
    model = model, tokenizer = tokenizer, train_dataset = ds,
    dataset_text_field = 'text', max_seq_length = MAX_SEQ_LEN,
    args = TrainingArguments(
        per_device_train_batch_size = 2, gradient_accumulation_steps = 4,
        warmup_steps = 5, max_steps = MAX_STEPS, learning_rate = LEARNING_RATE,
        fp16 = not torch.cuda.is_bf16_supported(), bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 1, optim = 'adamw_8bit', weight_decay = 0.01,
        lr_scheduler_type = 'linear', seed = 3407, output_dir = 'outputs'))
trainer.train()

In [ ]:
# 7) Export. Save the LoRA adapter, and a GGUF you can import into Ollama.
model.save_pretrained(OUTPUT_NAME); tokenizer.save_pretrained(OUTPUT_NAME)
# GGUF for Ollama (q4_k_m). Comment out if you only want the adapter.
model.save_pretrained_gguf(OUTPUT_NAME + '-gguf', tokenizer, quantization_method = 'q4_k_m')
print('Done. Download the folder, then on your host:')
print(f"  ollama create {OUTPUT_NAME} -f Modelfile   # Modelfile: FROM ./{OUTPUT_NAME}-gguf/*.gguf")
print('Then point LocalLLM at it (MODEL_NAME=' + OUTPUT_NAME + ', INFERENCE_BACKEND=ollama).')